In [1]:
import torch
import torch.nn as nn
import torch.optim as optim 

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# datasets and dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#image=>scale[0,1]=>normalize[-1,1]
transform=transforms.Compose([
    transforms.ToTensor(), #ToTensor=>tranform to tensore plus scale tensors
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset=CIFAR10(root="./data",train=True,download=True,transform=transform)
testset=CIFAR10(root="./data",train=False,download=True,transform=transform)

In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
train_loader=DataLoader(trainset,batch_size=64,shuffle=True)
test_loader=DataLoader(testset,batch_size=64)

### build the cnn

In [25]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
            nn.Conv2d(3, 32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2) ,#kernel size=2,stride=2
        
            nn.Conv2d(32, 64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )


    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1) #flattens

        x=self.fc_layers(x)

        return x

In [26]:
model=CNN()

In [27]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

# training the cnn

In [31]:
epochs=10

for epoch in range(epochs):
    epoch_training_loss=0.0

    for images,label in train_loader:
        optimizer.zero_grad()
        
        ouput=model.forward(images)#fp
        loss=criterion(ouput,label) #loss fnx
        loss.backward()
        optimizer.step() #update params

        epoch_training_loss+=loss.item()

    print(f"epoch={epoch+1}/{epochs} and loss={epoch_training_loss/len(train_loader)}")

epoch=1/10 and loss=0.7900527784281679
epoch=2/10 and loss=0.659949089407616
epoch=3/10 and loss=0.5484483429538015
epoch=4/10 and loss=0.4589899057508125
epoch=5/10 and loss=0.3747856980737518
epoch=6/10 and loss=0.3009912197852074
epoch=7/10 and loss=0.23427477666674673
epoch=8/10 and loss=0.18604626355554593
epoch=9/10 and loss=0.15021010608795812
epoch=10/10 and loss=0.12166917683494746


# evaluate our cnn

In [36]:
correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in test_loader:
        ouput=model.forward(images)
        _,predicted=torch.max(ouput,1)

        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)

print(f"accuracy={correct_labels/total_labels*100}")

accuracy=74.96000000000001
